Part I

In [5]:
import math

def solve_lp(c, constraints):
    """
    Solves a 2-variable LP:
      maximize: c[0]*x1 + c[1]*x2 
      subject to each constraint: a*x1 + b*x2 <= rhs
    using vertex enumeration.
    
    Parameters:
      c: tuple (c1, c2)
      constraints: list of tuples (a, b, rhs) representing: a*x1 + b*x2 <= rhs
                   (nonnegativity constraints should be included as (-1,0,0) for x1>=0 and (0,-1,0) for x2>=0)
    
    Returns:
      (opt_val, (x1, x2)) where opt_val is the maximum objective value and (x1, x2) is the vertex achieving it.
      If the feasible region is empty, returns (None, None).
    """
    # Gather potential vertices (intersection points)
    vertices = []
    n = len(constraints)
    
    # Compute intersections between every pair of constraints
    for i in range(n):
        a1, b1, rhs1 = constraints[i]
        for j in range(i+1, n):
            a2, b2, rhs2 = constraints[j]
            det = a1 * b2 - a2 * b1
            if abs(det) < 1e-9:
                continue  # parallel or nearly so, skip
            x1 = (rhs1 * b2 - rhs2 * b1) / det
            x2 = (a1 * rhs2 - a2 * rhs1) / det
            vertices.append((x1, x2))
    
    # Also consider intersections with the coordinate axes.
    # For each constraint, set x1=0 (if b != 0) and x2=0 (if a != 0)
    for (a, b, rhs) in constraints:
        if abs(b) > 1e-9:
            x2 = rhs / b
            vertices.append((0, x2))
        if abs(a) > 1e-9:
            x1 = rhs / a
            vertices.append((x1, 0))
    
    # Also include the origin
    vertices.append((0, 0))
    
    # Filter out vertices that are not feasible
    feasible = []
    for x in vertices:
        if all(a*x[0] + b*x[1] <= rhs + 1e-6 for (a, b, rhs) in constraints):
            feasible.append(x)
    
    if not feasible:
        return None, None
    
    # Evaluate objective function at each feasible vertex
    best_val = -float('inf')
    best_vertex = None
    for x in feasible:
        val = c[0]*x[0] + c[1]*x[1]
        if val > best_val:
            best_val = val
            best_vertex = x
    return best_val, best_vertex

def simplex(c1, c2, a11, a12, a21, a22, b1, b2):
    """
    Solves the following LP:
    
      Maximize: z = c1*x1 + c2*x2
      Subject to:
         a11*x1 + a12*x2 <= b1,
         a21*x1 + a22*x2 <= b2,
         x1, x2 >= 0.
         
    Parameters:
      c1, c2: coefficients in the objective function.
      a11, a12: coefficients of the first constraint.
      a21, a22: coefficients of the second constraint.
      b1, b2: right-hand side values for the constraints.
      
    Returns:
      (z, (x1, x2)) where z is the optimal objective value and (x1, x2) is the optimal solution.
    """
    # Define the constraint list including nonnegativity constraints:
    constraints = [
        (a11, a12, b1),
        (a21, a22, b2),
        (-1, 0, 0),   # represents x1 >= 0  (i.e., -x1 <= 0)
        (0, -1, 0)    # represents x2 >= 0  (i.e., -x2 <= 0)
    ]
    return solve_lp((c1, c2), constraints)

# Test using the class example:
# Maximize: z = 100*x1 + 150*x2
# Subject to:
#    3*x1 + 2*x2 <= 6,
#    1*x1 + 2*x2 <= 5,
#    x1, x2 >= 0.
if __name__ == "__main__":
    optimal_z, optimal_solution = simplex(100, 150, 3, 2, 1, 2, 6, 5)
    print("Optimal z =", optimal_z)
    print("Optimal (x1, x2) =", optimal_solution)

Optimal z = 387.5
Optimal (x1, x2) = (0.5, 2.25)


Part II

In [6]:
import math

def solve_lp(c, constraints):
    """
    Solves a 2-variable LP:
      maximize: c[0]*x1 + c[1]*x2 
      subject to each constraint: a*x1 + b*x2 <= rhs
    using vertex enumeration.
    
    Parameters:
      c: tuple (c1, c2)
      constraints: list of tuples (a, b, rhs) representing: a*x1 + b*x2 <= rhs
                   (nonnegativity constraints should be included as (-1,0,0) for x1>=0 and (0,-1,0) for x2>=0)
    
    Returns:
      (opt_val, (x1, x2)) where opt_val is the maximum objective value and (x1, x2) is the vertex achieving it.
      If the feasible region is empty, returns (None, None).
    """
    # Gather potential vertices (intersection points)
    vertices = []
    n = len(constraints)
    
    # Compute intersections between every pair of constraints
    for i in range(n):
        a1, b1, rhs1 = constraints[i]
        for j in range(i+1, n):
            a2, b2, rhs2 = constraints[j]
            det = a1 * b2 - a2 * b1
            if abs(det) < 1e-9:
                continue  # parallel or nearly so, skip
            x1 = (rhs1 * b2 - rhs2 * b1) / det
            x2 = (a1 * rhs2 - a2 * rhs1) / det
            vertices.append((x1, x2))
    
    # Also consider intersections with the coordinate axes.
    # For each constraint, set x1=0 (if b != 0) and x2=0 (if a != 0)
    for (a, b, rhs) in constraints:
        if abs(b) > 1e-9:
            x2 = rhs / b
            vertices.append((0, x2))
        if abs(a) > 1e-9:
            x1 = rhs / a
            vertices.append((x1, 0))
    
    # Also include the origin
    vertices.append((0, 0))
    
    # Filter out vertices that are not feasible
    feasible = []
    for x in vertices:
        if all(a*x[0] + b*x[1] <= rhs + 1e-6 for (a,b,rhs) in constraints):
            feasible.append(x)
    
    if not feasible:
        return None, None
    
    # Evaluate objective function
    best_val = -float('inf')
    best_vertex = None
    for x in feasible:
        val = c[0]*x[0] + c[1]*x[1]
        if val > best_val:
            best_val = val
            best_vertex = x
    return best_val, best_vertex

def branch_and_bound(c, base_constraints):
    """
    Uses branch-and-bound to solve the integer programming problem
      maximize: c[0]*x1 + c[1]*x2 
      subject to: base_constraints (a list of (a,b,rhs) inequalities)
                  and x1, x2 integer.
    
    Parameters:
      c: tuple (c1, c2)
      base_constraints: list of constraints (including original and nonnegativity)
    
    Returns:
      (best_obj, (x1, x2)) for the integer optimal solution.
    """
    best_obj = -float('inf')
    best_sol = None

    def bb(constraints):
        nonlocal best_obj, best_sol
        lp_val, lp_x = solve_lp(c, constraints)
        if lp_x is None:
            return  # infeasible
        # If the LP optimum is not better than current best, prune.
        if lp_val <= best_obj + 1e-6:
            return
        # Check if solution is integer
        x1, x2 = lp_x
        if abs(x1 - round(x1)) < 1e-6 and abs(x2 - round(x2)) < 1e-6:
            if lp_val > best_obj:
                best_obj = lp_val
                best_sol = (int(round(x1)), int(round(x2)))
            return
        
        # Branch on the variable with larger fractional part
        frac_x1 = abs(x1 - round(x1))
        frac_x2 = abs(x2 - round(x2))
        if frac_x1 >= frac_x2:
            var_index = 0
            frac_val = x1
        else:
            var_index = 1
            frac_val = x2
        
        floor_val = math.floor(frac_val)
        ceil_val = floor_val + 1
        
        # Branch 1: add constraint: x[var_index] <= floor_val
        if var_index == 0:
            new_constraint = (1, 0, floor_val)
        else:
            new_constraint = (0, 1, floor_val)
        bb(constraints + [new_constraint])
        
        # Branch 2: add constraint: x[var_index] >= ceil_val, equivalently -x[var_index] <= -ceil_val
        if var_index == 0:
            new_constraint = (-1, 0, -ceil_val)
        else:
            new_constraint = (0, -1, -ceil_val)
        bb(constraints + [new_constraint])
    
    bb(base_constraints)
    return best_obj, best_sol

def integer_programming(c1, c2, a11, a12, a21, a22, b1, b2):
    """
    Solves the following integer programming model:
    
       maximize: z = c1*x1 + c2*x2
       subject to:
           a11*x1 + a12*x2 <= b1,
           a21*x1 + a22*x2 <= b2,
           x1, x2 >= 0 and integer.
    
    Parameters are the coefficients.
    
    Returns:
       (z, (x1, x2)) the optimal objective value and solution.
    """
    c = (c1, c2)
    # Define the base constraints.
    base_constraints = [
        (a11, a12, b1),
        (a21, a22, b2),
        # Nonnegativity constraints: x1 >= 0 and x2 >= 0 written as -x1 <= 0, -x2 <= 0.
        (-1, 0, 0),
        (0, -1, 0)
    ]
    return branch_and_bound(c, base_constraints)

# Test using the class example:
# Maximize: z = 100*x1 + 150*x2
# Subject to: 3*x1 + 2*x2 <= 6,   x1 + 2*x2 <= 5, and x1, x2 are nonnegative integers.
if __name__ == "__main__":
    z, sol = integer_programming(100, 150, 3, 2, 1, 2, 6, 5)
    print("Optimal z =", z)
    print("Optimal (x1, x2) =", sol)

Optimal z = 300.0
Optimal (x1, x2) = (0, 2)


Part III

In [7]:
# pip install ortools
from ortools.linear_solver import pywraplp

def main():
    # Create the SCIP solver.
    solver = pywraplp.Solver.CreateSolver('SCIP')
    if not solver:
        print("SCIP solver is unavailable.")
        return

    # Data for warehouses and cities.
    capacities = [500, 1800]         # Warehouse capacities: Warehouse #1 and #2.
    demands = [600, 700, 300]          # Demands for City #1, #2, and #3.
    
    # Transportation costs from each warehouse to each city.
    costs = [
        [2, 1.5, 10],  # Costs from Warehouse #1 to City #1, City #2, and City #3.
        [4, 3.5, 6]    # Costs from Warehouse #2 to City #1, City #2, and City #3.
    ]
    
    num_warehouses = len(capacities)
    num_cities = len(demands)
    
    # Define decision variables: x[i][j] is the number of units shipped from warehouse i to city j.
    x = {}
    for i in range(num_warehouses):
        for j in range(num_cities):
            x[i, j] = solver.NumVar(0, solver.infinity(), f'x_{i}_{j}')
    
    # Supply constraints: shipments from each warehouse cannot exceed its capacity.
    for i in range(num_warehouses):
        solver.Add(sum(x[i, j] for j in range(num_cities)) <= capacities[i])
    
    # Demand constraints: each city's demand must be exactly met.
    for j in range(num_cities):
        solver.Add(sum(x[i, j] for i in range(num_warehouses)) == demands[j])
    
    # Define the objective function: minimize total transportation cost.
    objective = solver.Objective()
    for i in range(num_warehouses):
        for j in range(num_cities):
            objective.SetCoefficient(x[i, j], costs[i][j])
    objective.SetMinimization()
    
    # Solve the problem.
    result_status = solver.Solve()
    
    if result_status == pywraplp.Solver.OPTIMAL:
        print("Solution:")
        print("Minimum total cost =", objective.Value())
        print("Optimal shipment quantities:")
        for i in range(num_warehouses):
            for j in range(num_cities):
                print(f"  Warehouse #{i+1} -> City #{j+1}: {x[i, j].solution_value()}")
    else:
        print("The problem does not have an optimal solution.")

if __name__ == '__main__':
    main()

Solution:
Minimum total cost = 5650.0
Optimal shipment quantities:
  Warehouse #1 -> City #1: 0.0
  Warehouse #1 -> City #2: 500.0
  Warehouse #1 -> City #3: 0.0
  Warehouse #2 -> City #1: 600.0
  Warehouse #2 -> City #2: 200.0
  Warehouse #2 -> City #3: 300.0
